# PyTorch DDP Fashion MNIST Training Example

This example demonstrates how to train a convolutional neural network (CNN) to classify images using the [Fashion MNIST](https://github.com/zalandoresearch/fashion-mnist) dataset and [PyTorch Distributed Data Parallel (DDP)](https://pytorch.org/tutorials/intermediate/ddp_tutorial.html).

This notebook walks you through running that example in a local notebook resources, and how to easily scale PyTorch DDP across multiple nodes with Kubeflow TrainJob.

The reference for the original notebook is [here](https://github.com/kubeflow/trainer/blob/master/examples/pytorch/image-classification/mnist.ipynb)

## Verification of dependencies and their versions

In [1]:
import kubeflow
import torch
import torchvision
print(f"Kubeflow version: {kubeflow.__version__ if hasattr(kubeflow, '__version__') else 'N/A'}")
print(f"PyTorch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")

print("All imports successful!")

Kubeflow version: 0.3.0
PyTorch version: 2.3.1+cu121
torchvision version: 0.18.1+cu121
All imports successful!


## Define the Training Function

Create function to train CNN model using Fashion MNIST data along with their config values (number of samples, epochs, etc)

In [2]:
def train_fashion_mnist():
    import os
    import random

    import torch
    import torch.distributed as dist
    import torch.nn.functional as F
    from torch import nn
    from torch.utils.data import DataLoader, DistributedSampler, Subset
    from torchvision import datasets, transforms

    # Define the PyTorch CNN model to be trained
    class Net(nn.Module):
        def __init__(self):
            super(Net, self).__init__()
            self.conv1 = nn.Conv2d(1, 20, 5, 1)
            self.conv2 = nn.Conv2d(20, 50, 5, 1)
            self.fc1 = nn.Linear(4 * 4 * 50, 500)
            self.fc2 = nn.Linear(500, 10)

        def forward(self, x):
            x = F.relu(self.conv1(x))
            x = F.max_pool2d(x, 2, 2)
            x = F.relu(self.conv2(x))
            x = F.max_pool2d(x, 2, 2)
            x = x.view(-1, 4 * 4 * 50)
            x = F.relu(self.fc1(x))
            x = self.fc2(x)
            return F.log_softmax(x, dim=1)

    # Use NCCL if a GPU is available, otherwise use Gloo as communication backend.
    device, backend = ("cuda", "nccl") if torch.cuda.is_available() else ("cpu", "gloo")
    print(f"Using Device: {device}, Backend: {backend}")

    # Setup PyTorch distributed.
    local_rank = int(os.getenv("LOCAL_RANK", 0))
    dist.init_process_group(backend=backend)
    print(
        "Distributed Training for WORLD_SIZE: {}, RANK: {}, LOCAL_RANK: {}".format(
            dist.get_world_size(),
            dist.get_rank(),
            local_rank,
        )
    )

    # Create the model and load it into the device.
    device = torch.device(f"{device}:{local_rank}")
    model = nn.parallel.DistributedDataParallel(Net().to(device))
    optimizer = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)

    
    # Download FashionMNIST dataset only on local_rank=0 process.
    if local_rank == 0:
        dataset = datasets.FashionMNIST(
            "./data",
            train=True,
            download=True,
            transform=transforms.Compose([transforms.ToTensor()]),
        )
    dist.barrier()
    dataset = datasets.FashionMNIST(
        "./data",
        train=True,
        download=False,
        transform=transforms.Compose([transforms.ToTensor()]),
    )

    # Create a random subset
    num_samples = 1000
    
    # Set seed for reproducibility
    random.seed(42)
    all_indices = list(range(len(dataset)))
    random.shuffle(all_indices)
    subset_indices = all_indices[:num_samples]
    subset_dataset = Subset(dataset, subset_indices)

    # Shard the dataset across workers.
    train_loader = DataLoader(
        subset_dataset,
        batch_size=100,
        sampler=DistributedSampler(subset_dataset),
        num_workers=0  # Set to 0 to avoid multiprocessing issues
    )

    EPOCHS=2

    # TODO(astefanutti): add parameters to the training function
    dist.barrier()
    for epoch in range(1, EPOCHS):
        model.train()

        # Iterate over mini-batches from the training set
        for batch_idx, (inputs, labels) in enumerate(train_loader):
            # Copy the data to the GPU device if available
            inputs, labels = inputs.to(device), labels.to(device)
            # Forward pass
            outputs = model(inputs)
            loss = F.nll_loss(outputs, labels)
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Only print progress every 20 batches
            # And only show actual loss, not percentage updates
            if batch_idx % 20 == 0 and dist.get_rank() == 0:
                print(
                    "Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f}".format(
                        epoch,
                        batch_idx * len(inputs),
                        len(subset_dataset),
                        100.0 * batch_idx / len(train_loader),
                        loss.item(),
                    )
                )

    # Wait for the distributed training to complete
    dist.barrier()
    if dist.get_rank() == 0:
        print("Training is finished")

    # Finally clean up PyTorch distributed
    dist.destroy_process_group()

## Scale PyTorch DDP with Kubeflow TrainJob

You can use `TrainerClient()` from the Kubeflow SDK to communicate with Kubeflow Trainer APIs and scale your training function across multiple PyTorch training nodes.

`TrainerClient()` verifies that you have required access to the Kubernetes cluster.

Kubeflow Trainer creates a `TrainJob` resource and automatically sets the appropriate environment variables to set up PyTorch in distributed environment.



In [3]:
from kubeflow.trainer import CustomTrainer, TrainerClient
client = TrainerClient()

## List the Training Runtimes

You can get the list of available Training Runtimes to start your TrainJob.

Additionally, it might show available accelerator type and number of available resources.

In [4]:
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:v2.1.0', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None)
Runtime(name='torchtune-llama3.2-1b', trainer=RuntimeTrainer(trainer_type=<TrainerType.BUILTIN_TRAINER: 'BuiltinTrainer'>, framework='torchtune', image='ghcr.io/kubeflow/trainer/torchtune-trainer:

## Run the Distributed TrainJob

Kubeflow TrainJob will train the above model on PyTorch nodes defined by `NUM_NODES` and each node with `RESOURCES_PER_NODE`.

In [5]:
# Set your distributed environment configuration here

## Set how many PyTorch nodes you want to use for distributed training.
NUM_NODES = 5

# Set the resources for each PyTorch node.
RESOURCES_PER_NODE = {
    "cpu": "5",           # CPUs per node
    "memory": "2Gi",     # Memory in GiB per node
    "nvidia.com/gpu": 1,  # GPUs per node (the number will depend on the available resources)
}

In [6]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_fashion_mnist,
        num_nodes=NUM_NODES,
        resources_per_node=RESOURCES_PER_NODE,
    ),
    runtime=torch_runtime,
)

In [7]:
#Check job status directly
job = client.get_job(job_name)
print(f"\nJob ID: {job_name}")
print(f"Job Status: {job.status}")
print(f"Creation Time: {job.creation_timestamp}")
print(f"\nJob details: {job}")


Job ID: v54b375a0eb1
Job Status: Created
Creation Time: 2026-01-28 13:05:09+00:00

Job details: TrainJob(name='v54b375a0eb1', runtime=Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.7.1-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None), steps=[], num_nodes=5, creation_timestamp=datetime.datetime(2026, 1, 28, 13, 5, 9, tzinfo=TzInfo(0)), status='Created')


In [8]:
from datetime import datetime
import time
print("Waiting for job logs...")
wait_count = 0

while True:
    initial_logs = list(client.get_job_logs(job_name, follow=True))
    if initial_logs:
        print(f"Logs received after {wait_count} seconds:")
        for log in initial_logs:
            print(f"  {log}")
        break
    
    wait_count += 1
    print(f"[{datetime.now().strftime('%H:%M:%S')}] Waiting... ({wait_count}s)")
    time.sleep(1)

Waiting for job logs...
Logs received after 0 seconds:
  Using Device: cuda, Backend: nccl
  Distributed Training for WORLD_SIZE: 5, RANK: 0, LOCAL_RANK: 0
100%|██████████| 26.4M/26.4M [00:00<00:00, 38.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 2.24MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 28.4MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 36.2MB/s]
  /opt/conda/lib/python3.11/site-packages/torch/distributed/distributed_c10d.py:4631: UserWarning: No device id is provided via `init_process_group` or `barrier `. Using the current device set by the user. 
    warnings.warn(  # warn only once
  Train Epoch: 1 [0/1000 (0%)]	Loss: 2.303324
  Training is finished


# Delete the TrainJob
When TrainJob is finished, you can delete the resource.

In [9]:
client.delete_job(job_name)

# Run the Training in local notebook resource
We can submit the training function to the local Trainer client to run it in an isolated subprocess.

In [10]:
from kubeflow.trainer import CustomTrainer, TrainerClient, LocalProcessBackendConfig
from datetime import datetime

# Initialize local backend
backend_config = LocalProcessBackendConfig(cleanup_venv=True)
client = TrainerClient(backend_config=backend_config)

# List available runtimes
for runtime in client.list_runtimes():
    if runtime.name == "torch-distributed":
        torch_runtime = runtime
        break

# Submit training job
job_name = client.train(
    trainer=CustomTrainer(
        func=train_fashion_mnist,
        packages_to_install=["torch", "torchvision"],
    ),
    runtime=torch_runtime,
)

# Stream only training logs
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end='')

Operating inside /tmp/ke7d783e0bee86i7ml4v
Looking in links: /tmp/tmphf4jrfmc
Processing /tmp/tmphf4jrfmc/setuptools-65.5.0-py3-none-any.whl
Processing /tmp/tmphf4jrfmc/pip-24.0-py3-none-any.whl
  Using cached torch-2.10.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached torchvision-0.25.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.4 kB)
  Using cached filelock-3.20.3-py3-none-any.whl.metadata (2.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached cuda_bindings-12.9.4-cp311-cp311-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.6 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  